# Sesión 1 — Representación de problemas y búsqueda no informada


In [1]:
import random
import heapq
import math
import time
from collections import deque, Counter
from itertools import combinations

class Problem(object):
    """Clase abstracta para un problema de búsqueda (AIMA-Python)."""
    def __init__(self, initial=None, goal=None, **kwds):
        self.__dict__.update(initial=initial, goal=goal, **kwds)
    def actions(self, state): raise NotImplementedError
    def result(self, state, action): raise NotImplementedError
    def is_goal(self, state): return state == self.goal
    def action_cost(self, s, a, s1): return 1
    def h(self, node): return 0
    def __str__(self): return '{}({!r}, {!r})'.format(type(self).__name__, self.initial, self.goal)

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.__dict__.update(state=state, parent=parent, action=action, path_cost=path_cost)
    def __repr__(self): return '<{}>'.format(self.state)
    def __len__(self): return 0 if self.parent is None else 1 + len(self.parent)
    def __lt__(self, other): return self.path_cost < other.path_cost

failure = Node('failure', path_cost=math.inf)
cutoff = Node('cutoff', path_cost=math.inf)

def expand(problem, node):
    """Expande un nodo generando sus sucesores."""
    s = node.state
    for action in problem.actions(s):
        s1 = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, s1)
        yield Node(s1, node, action, cost)

def path_actions(node):
    if node.parent is None: return []
    return path_actions(node.parent) + [node.action]

def path_states(node):
    if node in (cutoff, failure, None): return []
    return path_states(node.parent) + [node.state]

FIFOQueue = deque
LIFOQueue = list

class PriorityQueue:
    def __init__(self, items=(), key=lambda x: x):
        self.key = key
        self.items = []
        self.counter = 0
        for item in items:
            self.add(item)

    def add(self, item):
        heapq.heappush(self.items, (self.key(item), self.counter, item))
        self.counter += 1

    def pop(self):
        return heapq.heappop(self.items)[2]

    def top(self):
        return self.items[0][2]

    def f_top(self):
        return self.items[0][0]

    def __len__(self):
        return len(self.items)


In [2]:
def best_first_search(problem, f):
    "Search nodes with minimum f(node) value first."
    node = Node(problem.initial)
    frontier = PriorityQueue([node], key=f)
    reached = {problem.initial: node}

    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
              
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                frontier.add(child)

    return failure


def best_first_tree_search(problem, f):
    "A version of best_first_search without the `reached` table."

    frontier = PriorityQueue([Node(problem.initial)], key=f)

    while frontier:
        node = frontier.pop()

        if problem.is_goal(node.state):
            return node

        for child in expand(problem, node):
            if not is_cycle(child):
                frontier.add(child)

    return failure


def g(n): return n.path_cost

def uniform_cost(problem):
    return best_first_search(problem, f=g)

def breadth_first(problem):
    return best_first_search(problem, f=len)

def depth_first(problem):
    return best_first_search(problem, f=lambda n: -len(n))

def breadth_first_tree(problem):
    return best_first_tree_search(problem, f=len)

def depth_first_tree(problem):
    return best_first_tree_search(problem, f=lambda n: -len(n))

def uniform_cost_tree(problem):
    return best_first_tree_search(problem, f=g)

def is_cycle(node, k=30):
    def find_cycle(ancestor, k):
        return (ancestor is not None and k > 0 and
                (ancestor.state == node.state or find_cycle(ancestor.parent, k - 1)))
    return find_cycle(node.parent, k)


## 8-puzzle

In [3]:
class EightPuzzle(Problem):
    def __init__(self, initial, goal=(1, 2, 3, 8, 0, 4, 7, 6, 5)):
        assert inversions(initial) % 2 == inversions(goal) % 2, \
            "El estado inicial y el objetivo no son alcanzables 8-puzzle."
        self.initial, self.goal = tuple(initial), tuple(goal)

    def actions(self, state):
        moves = ((1, 3), (0, 2, 4), (1, 5),
                 (0, 4, 6), (1, 3, 5, 7), (2, 4, 8),
                 (3, 7), (4, 6, 8), (7, 5))
        return moves[state.index(0)]

    def result(self, state, action):
        s = list(state)
        blank = state.index(0)
        s[action], s[blank] = s[blank], s[action]
        return tuple(s)

def inversions(board):
    return sum((a > b and a != 0 and b != 0) for a, b in combinations(board, 2))

def board8(board, fmt=(3 * '{} {} {}\n')):
    return fmt.format(*board).replace('0', '_')


In [4]:
ejemplos_8puzzle = {
 5: [(1,0,3,8,2,5,7,4,6),(1,4,2,0,8,3,7,6,5),(1,3,4,8,6,2,7,0,5),(1,2,3,7,8,0,6,5,4)],
10: [(8,2,1,7,0,3,6,5,4),(1,4,0,8,5,2,7,3,6),(8,1,3,7,0,5,4,2,6),(8,1,2,4,0,6,7,5,3)],
15: [(4,8,2,6,3,5,1,0,7),(1,4,5,2,7,0,8,6,3),(1,3,8,6,7,4,2,0,5),(2,0,8,7,5,3,4,1,6)],
20: [(6,2,7,4,5,1,0,8,3),(4,7,2,1,0,6,3,5,8),(7,1,5,4,0,8,2,6,3),(5,1,6,4,0,3,8,7,2)],
25: [(6,7,4,0,5,1,3,2,8),(6,0,7,5,4,1,3,8,2),(3,4,8,5,7,1,6,0,2)],
30: [(5,6,7,2,8,4,0,3,1),(5,6,7,4,0,8,3,2,1),(5,4,7,6,0,3,8,2,1)]
}
problemas_8puzzle = {d: [EightPuzzle(s) for s in estados]
                     for d, estados in ejemplos_8puzzle.items()}


## N-reinas


In [5]:
class NQueensProblem(Problem):
    """Coloca N reinas por columnas, de izquierda a derecha."""
    def __init__(self, N):
        self.N = N
        self.initial = tuple([None] * N)

    def actions(self, state):
        if state[-1] is not None: return []
        col = state.index(None)
        return [row for row in range(self.N) if not self.conflicted(state, row, col)]
    
    def result(self, state, row):
        col = state.index(None)
        new = list(state); new[col] = row
        return tuple(new)
    
    def conflicted(self, state, row, col):
        return any(self.conflict(row, col, state[c], c) for c in range(col))
    
    @staticmethod
    def conflict(row1, col1, row2, col2):
        return (row1 == row2 or row1-col1 == row2-col2 or row1+col1 == row2+col2)
    
    def is_goal(self, state): 
        return state[-1] is not None


## TSP

In [6]:
class EstadoTSP:
    def __init__(self, actual, visitadas):
        self.actual = actual
        self.visitadas = tuple(sorted(visitadas))

    def __hash__(self):
        return hash((self.actual, self.visitadas))

    def __eq__(self, other):
        return (isinstance(other, EstadoTSP)
                and self.actual == other.actual
                and self.visitadas == other.visitadas)

    def __repr__(self):
        return f'[Visitadas: {list(self.visitadas)} Actual: {self.actual}]'


class Grafo:
    def __init__(self, lista):
        self.N = int((((8 * len(lista) + 1) ** 0.5) - 1) / 2)
        self.ciudades = list(range(self.N))
        self.Dist = [[0] * self.N for _ in self.ciudades]

        acc = 0
        for x in range(self.N):
            acc += x
            for y in range(x + 1):
                self.Dist[x][y] = self.Dist[y][x] = lista[acc + y]


class SymmetricTSP(Problem):
    def __init__(self, grafo):
        self.grafo = grafo
        self.initial = EstadoTSP(0, ())

    def actions(self, state):
        if state.actual == -1:
            return []

        no_visitadas = [
            c for c in self.grafo.ciudades
            if c != state.actual and c not in state.visitadas
        ]

        return no_visitadas if no_visitadas else [-1]

    def result(self, state, action):
        visitadas = set(state.visitadas)
        visitadas.add(state.actual)
        return EstadoTSP(action, visitadas)

    def is_goal(self, state):
        return state.actual == -1

    def action_cost(self, s, a, s1):
        return self.grafo.Dist[s.actual][0] if a == -1 else self.grafo.Dist[s.actual][a]


In [7]:
# Instancias TSP
# La clase Grafo recibe la matriz simétrica almacenada por filas
# utilizando únicamente el triángulo inferior, incluida la diagonal.

ejemploClase = [
    0,
    21,0,
    12,7,0,
    15,32,5,0,
    113,25,18,180,0,
    92,9,20,39,17,0
]

gr17 = [
    0,633,0,257,390,0,91,661,228,0,412,227,169,383,0,150,488,112,120,267,0,
    80,572,196,77,351,63,0,134,530,154,105,309,34,29,0,259,555,372,175,338,264,232,249,0,
    505,289,262,476,196,360,444,402,495,0,353,282,110,324,61,208,292,250,352,154,0,
    324,638,437,240,421,329,297,314,95,578,435,0,70,567,191,27,346,83,47,68,189,439,287,254,0,
    211,466,74,182,243,105,150,108,326,336,184,391,145,0,268,420,53,239,199,123,207,165,383,240,140,448,202,57,0,
    246,745,472,237,528,364,332,349,202,685,542,157,289,426,483,0,
    121,518,142,84,297,35,29,36,236,390,238,301,55,96,153,336,0
]

gr21 = [
    0,510,0,635,355,0,91,415,605,0,385,585,390,350,0,155,475,495,120,240,0,
    110,480,570,78,320,96,0,130,500,540,97,285,36,29,0,490,605,295,460,120,350,425,390,0,
    370,320,700,280,590,365,350,370,625,0,155,380,640,63,430,200,160,175,535,240,0,
    68,440,575,27,320,91,48,67,430,300,90,0,610,360,705,520,835,605,590,610,865,250,480,545,0,
    655,235,585,555,750,615,625,645,775,285,515,585,190,0,480,81,435,380,575,440,455,465,600,245,345,415,295,170,0,
    265,480,420,235,125,125,200,165,230,475,310,205,715,650,475,0,
    255,440,755,235,650,370,320,350,680,150,175,265,400,435,385,485,0,
    450,270,625,345,660,430,420,440,690,77,310,380,180,215,190,545,225,0,
    170,445,750,160,495,265,220,240,600,235,125,170,485,525,405,375,87,315,0,
    240,290,590,140,480,255,205,220,515,150,100,170,390,425,255,395,205,220,155,0,
    380,140,495,280,480,340,350,370,505,185,240,310,345,280,105,380,280,165,305,150,0
]

# Grafos correspondientes
grafo_ejemplo = Grafo(ejemploClase)
grafo_gr17 = Grafo(gr17)
grafo_gr21 = Grafo(gr21)


## Experimentación


In [8]:
# 8-PUZZLE
p = problemas_8puzzle[15][1]

t0 = time.perf_counter()
sol = breadth_first(p)
t = time.perf_counter() - t0

print("BFS - búsqueda en grafo")
print("Coste:", sol.path_cost)
print("Tiempo:", t)
print("Acciones:", path_actions(sol))


BFS - búsqueda en grafo
Coste: 15
Tiempo: 0.0536034000001564
Acciones: [8, 7, 4, 5, 2, 1, 4, 3, 6, 7, 8, 5, 2, 1, 4]
